# Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_key AS customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'Unknown' THEN ci.gender
        ELSE COALESCE(ca.gender, 'Unknown')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM silver.crm_cust_info ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_key = ca.customer_number
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.customer_number
WHERE ci.customer_id IS NOT NULL
"""
df = spark.sql(query)

# Checking the DataFrame

In [0]:
df.limit(10).display()

# Writing into Gold Table

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.gold.dim_customers')
)